### What does the raw data look like?

In [2]:
from olist.ingestion import load_all
from olist.utils.profiling import profile

tables = load_all()

for name, df in tables.items():
    profile(df, name)

=== customers ===
Rows: 99,441 | Columns: 5
Duplicate rows: 0
                          dtype  missing  missing_%  unique
customer_id                 str        0        0.0   99441
customer_unique_id          str        0        0.0   96096
customer_zip_code_prefix  int64        0        0.0   14994
customer_city               str        0        0.0    4119
customer_state              str        0        0.0      27

=== geolocation ===
Rows: 1,000,163 | Columns: 5
Duplicate rows: 261,831
                               dtype  missing  missing_%  unique
geolocation_zip_code_prefix    int64        0        0.0   19015
geolocation_lat              float64        0        0.0  717360
geolocation_lng              float64        0        0.0  717613
geolocation_city                 str        0        0.0    8011
geolocation_state                str        0        0.0      27

=== order_items ===
Rows: 112,650 | Columns: 7
Duplicate rows: 0
                              dtype  missing  mi

In [3]:
orders = tables["orders"]

# Status of all orders
print(orders["order_status"].value_counts())

# Status of only the orders with no delivery date
print(orders[orders["order_delivered_customer_date"].isna()]["order_status"].value_counts())

order_status
delivered      96478
shipped         1107
canceled         625
unavailable      609
invoiced         314
processing       301
created            5
approved           2
Name: count, dtype: int64
order_status
shipped        1107
canceled        619
unavailable     609
invoiced        314
processing      301
delivered         8
created           5
approved          2
Name: count, dtype: int64


In [4]:
customers = tables["customers"]

# Find people who appear more than once
counts = customers["customer_unique_id"].value_counts()
print(counts.head())

customer_unique_id
8d50f5eadf50201ccdcedfb9e2ac8455    17
3e43e6105506432c953e165fb2acf44c     9
1b6c7548a2a1f9037c1fd3ddfed95f33     7
6469f99c1f9dfae7733b25662e7f1782     7
ca77025e7201e3b30c44b472ff346268     7
Name: count, dtype: int64


In [6]:
some_id = "8d50f5eadf50201ccdcedfb9e2ac8455"
customers[customers["customer_unique_id"] == "8d50f5eadf50201ccdcedfb9e2ac8455"]

,customer_id,customer_unique_id,customer_zip_code_prefix,customer_city,customer_state
14186,1bd3585471932167ab72a84955ebefea,8d50f5eadf50201ccdcedfb9e2ac8455,4045,sao paulo,SP
15321,a8fabc805e9a10a3c93ae5bff642b86b,8d50f5eadf50201ccdcedfb9e2ac8455,4045,sao paulo,SP
16654,897b7f72042714efaa64ac306ba0cafc,8d50f5eadf50201ccdcedfb9e2ac8455,4045,sao paulo,SP
36122,b2b13de0770e06de50080fea77c459e6,8d50f5eadf50201ccdcedfb9e2ac8455,4045,sao paulo,SP
38073,42dbc1ad9d560637c9c4c1533746f86d,8d50f5eadf50201ccdcedfb9e2ac8455,4045,sao paulo,SP
40141,dfb941d6f7b02f57a44c3b7c3fefb44b,8d50f5eadf50201ccdcedfb9e2ac8455,4045,sao paulo,SP
48614,65f9db9dd07a4e79b625effa4c868fcb,8d50f5eadf50201ccdcedfb9e2ac8455,4045,sao paulo,SP
52574,1c62b48fb34ee043310dcb233caabd2e,8d50f5eadf50201ccdcedfb9e2ac8455,4045,sao paulo,SP
58707,a682769c4bc10fc6ef2101337a6c83c9,8d50f5eadf50201ccdcedfb9e2ac8455,4045,sao paulo,SP
67996,6289b75219d757a56c0cce8d9e427900,8d50f5eadf50201ccdcedfb9e2ac8455,4045,sao paulo,SP


In [7]:
import pandas as pd

people = pd.DataFrame({"id": [1, 2], "name": ["Rahul", "Priya"]})
orders_demo = pd.DataFrame({"order_no": ["A", "B", "C"], "id": [1, 1, 2]})

pd.merge(orders_demo, people, on="id")

,order_no,id,name
0,A,1,Rahul
1,B,1,Rahul
2,C,2,Priya


In [8]:
orders = tables["orders"]

merged = pd.merge(orders, customers, on="customer_id")

print(merged.shape)

(99441, 12)


In [9]:
some_id = "8d50f5eadf50201ccdcedfb9e2ac8455"

merged[merged["customer_unique_id"] == some_id][
    ["order_id", "customer_unique_id", "order_status", "order_purchase_timestamp"]
]

,order_id,customer_unique_id,order_status,order_purchase_timestamp
2340,c2213109a2cc0e75d55585b7aaac6d97,8d50f5eadf50201ccdcedfb9e2ac8455,delivered,2018-08-07 23:32:14
3660,23427a6bd9f8fd1b51f1b1e5cc186ab8,8d50f5eadf50201ccdcedfb9e2ac8455,delivered,2018-05-21 22:44:31
5167,e3071b7624445af6e4f3a1b23718667d,8d50f5eadf50201ccdcedfb9e2ac8455,delivered,2017-09-05 22:14:52
11694,d3582fd5ccccd9cb229a63dfb417c86f,8d50f5eadf50201ccdcedfb9e2ac8455,delivered,2018-08-20 19:14:26
16231,5837a2c844decae8a778657425f6d664,8d50f5eadf50201ccdcedfb9e2ac8455,unavailable,2017-07-17 22:11:13
19127,bf92c69b7cc70f7fc2c37de43e366173,8d50f5eadf50201ccdcedfb9e2ac8455,delivered,2017-07-24 22:11:50
24886,6bdf325f0966e3056651285c0aed5aad,8d50f5eadf50201ccdcedfb9e2ac8455,delivered,2018-05-22 23:08:55
33703,4f62d593acae92cea3c5662c76122478,8d50f5eadf50201ccdcedfb9e2ac8455,delivered,2017-07-18 23:10:58
39449,b850a16d8faf65a74c51287ef34379ce,8d50f5eadf50201ccdcedfb9e2ac8455,delivered,2017-11-22 20:01:53
58811,519203404f6116d406a970763ee75799,8d50f5eadf50201ccdcedfb9e2ac8455,delivered,2017-08-05 08:59:43


In [10]:
orders_per_customer = merged.groupby("customer_unique_id")["order_id"].count()

print(orders_per_customer.value_counts().sort_index())

repeat_pct = (orders_per_customer > 1).mean() * 100
print(f"Repeat customers: {repeat_pct:.2f}%")

order_id
1     93099
2      2745
3       203
4        30
5         8
6         6
7         3
9         1
17        1
Name: count, dtype: int64
Repeat customers: 3.12%


In [11]:
print(merged["order_status"].value_counts())

order_status
delivered      96478
shipped         1107
canceled         625
unavailable      609
invoiced         314
processing       301
created            5
approved           2
Name: count, dtype: int64


In [12]:
delivered = merged[merged["order_status"] == "delivered"]

orders_per_customer_d = delivered.groupby("customer_unique_id")["order_id"].count()

repeat_pct_d = (orders_per_customer_d > 1).mean() * 100
print(f"Customers with a delivered order: {len(orders_per_customer_d):,}")
print(f"Repeat customers (delivered only): {repeat_pct_d:.2f}%")

Customers with a delivered order: 93,358
Repeat customers (delivered only): 3.00%
